In [2]:
!pip install transformers torch pandas scikit-learn

In [19]:
import re
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch.nn.functional as F

def normalizar_texto(texto):
    # 1. Convertir a minúsculas
    texto = texto.lower()
    # 2. Eliminar caracteres especiales y números que no aportan sentimiento
    texto = re.sub(r'[^a-záéíóúñ\s]', '', texto)
    # 3. Eliminar espacios múltiples
    texto = re.sub(r'\s+', ' ', texto).strip()
    return texto

# Función de informe actualizada con normalización
def informe_ia_mejorado(parrafo_sucio, modelos_dict):
    # Aplicar tratamiento al párrafo antes del análisis
    parrafo_limpio = normalizar_texto(parrafo_sucio)

    print(f"--- TEXTO ORIGINAL: {parrafo_sucio[:50]}...")
    print(f"--- TEXTO NORMALIZADO: {parrafo_limpio[:50]}...\n")

    traduccion_emociones = {
        'others': 'Neutral', 'joy': 'Alegría', 'sadness': 'Tristeza',
        'anger': 'Enojo', 'surprise': 'Sorpresa', 'disgust': 'Disgusto', 'fear': 'Miedo'
    }

    for nombre, ruta in modelos_dict.items():
        print(f"\n{'='*15} {nombre} {'='*15}")

        tokenizer = AutoTokenizer.from_pretrained(ruta)
        model = AutoModelForSequenceClassification.from_pretrained(ruta, output_attentions=True)

        inputs = tokenizer(parrafo_limpio, return_tensors="pt", max_length=512, truncation=True)
        if "robert" in ruta.lower(): inputs.pop("token_type_ids", None)

        with torch.no_grad():
            outputs = model(**inputs)
            probs = F.softmax(outputs.logits, dim=1).flatten()
            atenciones = outputs.attentions[-1]
            pesos_medios = atenciones[0].mean(dim=0).mean(dim=0).numpy()
            tokens = tokenizer.convert_ids_to_tokens(inputs['input_ids'][0])

        # Mostrar Sentimientos
        etiquetas = model.config.id2label
        print("\nDISTRIBUCIÓN DE SENTIMIENTOS:")
        for i, p in enumerate(probs):
            label = etiquetas[i]
            label_humano = traduccion_emociones.get(label, label)
            print(f"- {label_humano}: {p.item()*100:.2f}%")

        # Mostrar Pesos (ahora serán palabras reales)
        print("\nPESOS DE ATENCIÓN (Palabras Clave):")
        importancia = sorted(zip(tokens, pesos_medios), key=lambda x: x[1], reverse=True)
        filtrados = [x for x in importancia if x[0] not in ['[CLS]', '[SEP]', '<s>', '</s>', '<pad>']][:5]
        for tok, peso in filtrados:
            print(f"  > {tok.replace(' ', '')}: {peso:.4f}")

# Ejecuta con tus modelos
modelos_finales = {
    "RoBERTuito_Emociones": "pysentimiento/robertuito-emotion-analysis",
    "BETO_Sentimiento": "finiteautomata/beto-sentiment-analysis",
    "BERT_Multilingüe": "nlptown/bert-base-multilingual-uncased-sentiment"
}

parrafo_prueba = "En esto, descubrieron treinta o cuarenta molinos de viento que hay en aquel campo; y, así como don Quijote los vio, dijo a su escudero: —La ventura va guiando nuestras cosas mejor de lo que acertáramos a desear, porque ves allí, amigo Sancho Panza, donde se descubren treinta, o pocos más, desaforados gigantes, con quien pienso hacer batalla y quitarles a todos las vidas..."
informe_ia_mejorado(parrafo_prueba, modelos_finales)

--- TEXTO ORIGINAL: En esto, descubrieron treinta o cuarenta molinos d...
--- TEXTO NORMALIZADO: en esto descubrieron treinta o cuarenta molinos de...


=============== RoBERTuito_Emociones ===============


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: pysentimiento/robertuito-emotion-analysis
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



DISTRIBUCIÓN DE SENTIMIENTOS:
- Neutral: 93.01%
- Alegría: 4.28%
- Tristeza: 1.75%
- Enojo: 0.17%
- Sorpresa: 0.61%
- Disgusto: 0.06%
- Miedo: 0.12%

PESOS DE ATENCIÓN (Palabras Clave):
  > ▁ves: 0.0078
  > jo: 0.0076
  > ▁vent: 0.0076
  > ▁san: 0.0075
  > ▁mol: 0.0070

=============== BETO_Sentimiento ===============


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: finiteautomata/beto-sentiment-analysis
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



DISTRIBUCIÓN DE SENTIMIENTOS:
- NEG: 49.83%
- NEU: 27.44%
- POS: 22.74%

PESOS DE ATENCIÓN (Palabras Clave):
  > allí: 0.0217
  > escu: 0.0177
  > pan: 0.0167
  > donde: 0.0147
  > ves: 0.0146

=============== BERT_Multilingüe ===============


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]


DISTRIBUCIÓN DE SENTIMIENTOS:
- 1 star: 3.21%
- 2 stars: 4.07%
- 3 stars: 10.86%
- 4 stars: 29.29%
- 5 stars: 52.57%

PESOS DE ATENCIÓN (Palabras Clave):
  > en: 0.0255
  > va: 0.0156
  > dijo: 0.0154
  > donde: 0.0136
  > como: 0.0128


In [21]:
import re
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForSequenceClassification

def normalizacion_profunda(texto):
    # Eliminar puntuación excesiva y dejar solo texto limpio
    texto = re.sub(r'[^\w\s]', '', texto.lower())
    return texto

def limpiar_token(token):
    # Eliminar caracteres especiales de control de BERT y RoBERTa
    return token.replace(' ', '').replace('##', '').replace('▁', '').replace('Ġ', '')

def informe_final_uninorte(parrafo, modelos_dict):
    texto_limpio = normalizacion_profunda(parrafo)

    for nombre, ruta in modelos_dict.items():
        print(f"\n{'='*20} MODELO: {nombre} {'='*20}")

        tokenizer = AutoTokenizer.from_pretrained(ruta)
        model = AutoModelForSequenceClassification.from_pretrained(ruta, output_attentions=True)

        inputs = tokenizer(texto_limpio, return_tensors="pt", max_length=512, truncation=True)
        if "robert" in ruta.lower(): inputs.pop("token_type_ids", None)

        with torch.no_grad():
            outputs = model(**inputs)
            probs = F.softmax(outputs.logits, dim=1).flatten()
            # Mecanismo de Atención (Pesos de la última capa)
            atenciones = outputs.attentions[-1]
            pesos = atenciones[0].mean(dim=0).mean(dim=0).numpy()
            tokens_sucios = tokenizer.convert_ids_to_tokens(inputs['input_ids'][0])

        # 1. EVALUACIÓN DE SENTIMIENTOS
        print("NIVEL DE SENTIMIENTO:")
        etiquetas = model.config.id2label
        for i, p in enumerate(probs):
            print(f"  - {etiquetas[i]}: {p.item()*100:.2f}%")

        # 2. PESOS DE LAS ORACIONES (PALABRAS CLAVE LIMPIAS)
        print("\nPESOS DE ATENCIÓN (Palabras con más carga):")
        # Limpiamos y filtramos tokens de control
        importancia = []
        for t, p in zip(tokens_sucios, pesos):
            t_limpio = limpiar_token(t)
            if t_limpio and t not in ['[CLS]', '[SEP]', '<s>', '</s>', '<pad>']:
                importancia.append((t_limpio, p))

        # Ordenar por peso
        importancia = sorted(importancia, key=lambda x: x[1], reverse=True)[:5]
        for tok, peso in importancia:
            print(f"  > {tok}: {peso:.4f}")

# Ejecución
modelos = {
    "RoBERTuito_Emociones": "pysentimiento/robertuito-emotion-analysis",
    "BETO_Sentimiento": "finiteautomata/beto-sentiment-analysis",
    "BERT_Multilingual": "nlptown/bert-base-multilingual-uncased-sentiment"
}

parrafo_quijote = "En esto, descubrieron treinta o cuarenta molinos de viento que hay en aquel campo; y, así como don Quijote los vio, dijo a su escudero: —La ventura va guiando nuestras cosas mejor de lo que acertáramos a desear, porque ves allí, amigo Sancho Panza, donde se descubren treinta, o pocos más, desaforados gigantes, con quien pienso hacer batalla y quitarles a todos las vidas..."
informe_final_uninorte(parrafo_quijote, modelos)


==================== MODELO: RoBERTuito_Emociones ====================


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: pysentimiento/robertuito-emotion-analysis
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


NIVEL DE SENTIMIENTO:
  - others: 93.01%
  - joy: 4.28%
  - sadness: 1.75%
  - anger: 0.17%
  - surprise: 0.61%
  - disgust: 0.06%
  - fear: 0.12%

PESOS DE ATENCIÓN (Palabras con más carga):
  > ves: 0.0078
  > jo: 0.0076
  > vent: 0.0076
  > san: 0.0075
  > mol: 0.0070

==================== MODELO: BETO_Sentimiento ====================


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: finiteautomata/beto-sentiment-analysis
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


NIVEL DE SENTIMIENTO:
  - NEG: 49.83%
  - NEU: 27.44%
  - POS: 22.74%

PESOS DE ATENCIÓN (Palabras con más carga):
  > allí: 0.0217
  > escu: 0.0177
  > pan: 0.0167
  > donde: 0.0147
  > ves: 0.0146

==================== MODELO: BERT_Multilingual ====================


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

NIVEL DE SENTIMIENTO:
  - 1 star: 3.21%
  - 2 stars: 4.07%
  - 3 stars: 10.86%
  - 4 stars: 29.29%
  - 5 stars: 52.57%

PESOS DE ATENCIÓN (Palabras con más carga):
  > en: 0.0255
  > va: 0.0156
  > dijo: 0.0154
  > donde: 0.0136
  > como: 0.0128
